# TDE 01 — Análise Exploratória de Dados
## Análise de Volatilidade de Ações — Optiver Realized Volatility Prediction

**Disciplina:** Análise Exploratória de Dados  
**Tema:** Análise de Volatilidade de Ações  
**Base:** Optiver Realized Volatility Prediction — Kaggle

---

### Objetivo desta etapa

Esta primeira etapa tem como objetivo realizar uma **Análise Exploratória de Dados (EDA)** sobre dados de mercado financeiro, investigando a variável de volatilidade realizada (`target`) e características associadas aos ativos e aos períodos observados.

O projeto final busca prever a volatilidade futura das ações. Entretanto, neste TDE o foco é compreender os dados antes da modelagem: verificar sua estrutura, qualidade, distribuições, diferenças entre ações, comportamento temporal, valores atípicos e possíveis relações entre variáveis.

> **Observação:** o enunciado da disciplina informa que a avaliação futura das previsões utilizará MAE. Neste notebook não treinamos modelos; concentramos a análise na etapa exploratória.

## 1. Contextualização

Todos os dias, ações de empresas de capital aberto são negociadas em grandes volumes. Notícias, resultados financeiros, alterações fiscais, mudanças econômicas e o comportamento dos investidores podem modificar rapidamente os preços.

A intensidade dessas oscilações pode ser representada pela **volatilidade**. Em finanças, uma forma comum de medir volatilidade é utilizar o desvio-padrão dos retornos logarítmicos observados em determinado período.

A volatilidade também é relevante no mercado de derivativos. Opções, por exemplo, possuem seu valor fortemente relacionado à expectativa de variação futura do preço do ativo. Assim, compreender os fatores associados à volatilidade pode auxiliar investidores, gestores de risco e participantes do mercado.

A base utilizada pertence à competição **Optiver Realized Volatility Prediction**, disponibilizada no Kaggle. Ela contém dados de livro de ofertas (*order book*), negócios executados (*trades*) e a volatilidade realizada futura de diferentes ações.

### Pergunta principal da EDA

**Como a volatilidade realizada varia entre diferentes ações e períodos, e quais padrões, valores atípicos e relações com características de mercado podem ser identificados nos dados?**

## 2. Preparação do ambiente

O código abaixo instala as bibliotecas necessárias. No Google Colab, execute as células na ordem.

Para baixar os dados diretamente do Kaggle:

1. Entre em sua conta do Kaggle.
2. Aceite as regras da competição **Optiver Realized Volatility Prediction**.
3. No Kaggle, gere o arquivo de credenciais `kaggle.json`.
4. Execute a célula abaixo e envie esse arquivo quando solicitado.

O download completo é grande. Após a extração, o notebook carregará inicialmente apenas `train.csv` e, quando possível, uma partição de uma ação dos arquivos Parquet.

In [ ]:
!pip -q install kaggle pyarrow

In [ ]:
# Upload da credencial do Kaggle
# Funciona tanto no Google Colab quanto em Jupyter local.
import os, shutil

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")

try:
    from google.colab import files
    uploaded = files.upload()  # selecione kaggle.json

    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Envie o arquivo kaggle.json gerado na sua conta do Kaggle.")

    shutil.copy("kaggle.json", kaggle_json_path)

except ImportError:
    # Fora do Google Colab: kaggle.json deve estar em ~/.kaggle/kaggle.json,
    # ou as credenciais devem estar nas variáveis de ambiente
    # KAGGLE_USERNAME e KAGGLE_KEY.
    tem_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    if not os.path.exists(kaggle_json_path) and not tem_env:
        raise FileNotFoundError(
            "Fora do Google Colab: coloque o arquivo kaggle.json em "
            f"{kaggle_json_path} ou defina as variáveis de ambiente "
            "KAGGLE_USERNAME e KAGGLE_KEY."
        )

if os.path.exists(kaggle_json_path):
    os.chmod(kaggle_json_path, 0o600)

print("Credencial configurada com sucesso.")


In [ ]:
# Download dos dados da competição
!mkdir -p /content/optiver
!kaggle competitions download -c optiver-realized-volatility-prediction -p /content/optiver

In [ ]:
# Extração
import glob
import zipfile
import os

zip_files = glob.glob("/content/optiver/*.zip")
print("Arquivos ZIP encontrados:", zip_files)

for z in zip_files:
    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall("/content/optiver")

print("Extração concluída.")

## 3. Importação das bibliotecas

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("Bibliotecas carregadas.")

## 4. Localização e carregamento do conjunto principal

O arquivo `train.csv` contém:

- `stock_id`: identificador da ação;
- `time_id`: identificador do intervalo temporal;
- `target`: volatilidade realizada calculada no período futuro associado à observação.

Cada linha representa, portanto, uma combinação **ação + intervalo temporal**, acompanhada do valor de volatilidade que futuramente será utilizado como variável-alvo em modelos de previsão.

In [ ]:
# Localiza automaticamente o train.csv
train_paths = glob.glob("/content/optiver/**/train.csv", recursive=True)

if not train_paths:
    raise FileNotFoundError("train.csv não foi encontrado. Verifique se o download e a extração foram concluídos.")

train_path = train_paths[0]
print("Arquivo:", train_path)

df = pd.read_csv(train_path)
df.head()

## 5. Conhecimento inicial dos dados

Nesta etapa verificamos dimensão, tipos, exemplos de registros e estatísticas descritivas.

In [ ]:
print(f"Número de linhas: {df.shape[0]:,}")
print(f"Número de colunas: {df.shape[1]}")
print("\nColunas:", list(df.columns))
print("\nTipos:")
display(df.dtypes.to_frame("tipo"))

display(df.head())

In [ ]:
# Estatísticas descritivas
display(df.describe().T)

### Interpretação

As estatísticas acima permitem avaliar a escala das variáveis e observar inicialmente a distribuição da volatilidade. Como `stock_id` e `time_id` são identificadores, médias desses códigos não possuem interpretação financeira direta. A variável de maior interesse nesta fase é `target`.

### Classificação funcional das variáveis

Embora `stock_id` e `time_id` estejam armazenados como números inteiros, eles não têm significado métrico:

- **`stock_id`** — variável **categórica nominal**: identifica a ação, mas a ordem/magnitude do número não tem significado (a ação 10 não é "maior" que a ação 2).
- **`time_id`** — variável **categórica/ordinal de tempo**: identifica um intervalo de negociação. Preserva uma ordem relativa consistente entre ações, mas não corresponde a uma data/hora real.
- **`target`** — variável **numérica contínua**: a volatilidade realizada, foco principal desta EDA.

Essa distinção importa porque estatísticas como média e desvio-padrão fazem sentido para `target`, mas não para `stock_id`/`time_id` enquanto identificadores.

In [ ]:
print(f"stock_id -> {df['stock_id'].nunique()} valores únicos | categórica (identificador nominal)")
print(f"time_id  -> {df['time_id'].nunique()} valores únicos | categórica/ordinal (identificador de tempo)")
print(f"target   -> variável numérica contínua (dtype: {df['target'].dtype})")

## 6. Qualidade dos dados

Vamos investigar valores ausentes, duplicações, valores inválidos e possíveis observações extremas.

In [ ]:
qualidade = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "percentual_ausente": (df.isna().mean() * 100).round(4),
    "unicos": df.nunique()
})

display(qualidade)

In [ ]:
duplicadas = df.duplicated().sum()
duplicadas_chave = df.duplicated(subset=["stock_id", "time_id"]).sum()

print("Linhas completamente duplicadas:", duplicadas)
print("Duplicações da chave stock_id + time_id:", duplicadas_chave)
print("Targets negativos:", (df["target"] < 0).sum())
print("Targets iguais a zero:", (df["target"] == 0).sum())

### Critérios de tratamento

- Valores ausentes devem ser investigados antes de qualquer remoção.
- Duplicações em `stock_id + time_id` podem indicar inconsistência, pois a combinação deve representar uma observação específica.
- Como volatilidade é uma medida de dispersão, valores negativos de `target` seriam inválidos.
- Valores muito altos de volatilidade não devem ser removidos automaticamente, pois podem representar períodos reais de forte instabilidade no mercado.

Abaixo aplicamos esses critérios aos dados efetivamente carregados e registramos a decisão tomada em cada caso.

In [ ]:
print("=== Diagnóstico de qualidade e decisões de tratamento ===")

# Ausentes
n_ausentes = df["target"].isna().sum()
if n_ausentes == 0:
    print("Ausentes: nenhum valor ausente em 'target'. Nenhuma imputação/remoção necessária.")
else:
    pct = n_ausentes / len(df) * 100
    print(f"Ausentes: {n_ausentes} valor(es) em 'target' ({pct:.4f}% das linhas).")
    print("Decisão: linhas com target ausente serão excluídas das estatísticas de volatilidade, "
          "pois não é possível calcular métricas com o alvo indefinido.")

# Duplicadas completas
if duplicadas > 0:
    print(f"\nDuplicadas completas: {duplicadas} linha(s). Exemplos:")
    display(df[df.duplicated(keep=False)].sort_values(["stock_id", "time_id"]).head(10))
    print("Decisão: linhas completamente duplicadas serão removidas, pois não agregam informação nova.")
else:
    print("\nDuplicadas completas: nenhuma encontrada.")

# Duplicadas na chave
if duplicadas_chave > 0:
    print(f"\nDuplicações na chave stock_id+time_id (sem serem duplicatas completas): {duplicadas_chave}.")
    print("Decisão: como cada combinação stock_id+time_id deveria ser única, esses casos serão "
          "investigados individualmente antes de decidir entre manter, corrigir ou remover.")
else:
    print("\nChave stock_id+time_id: única em todas as linhas, como esperado.")

# Valores negativos
n_negativos = (df["target"] < 0).sum()
if n_negativos > 0:
    print(f"\nValores negativos de target: {n_negativos}.")
    print("Decisão: fisicamente inválidos para uma medida de dispersão — serão investigados e, "
          "se confirmados como erro, removidos ou corrigidos.")
else:
    print("\nValores negativos de target: nenhum encontrado — consistente com a natureza não-negativa da volatilidade.")

## 7. Análise univariada da volatilidade (`target`)

In [ ]:
target_stats = df["target"].agg([
    "count", "mean", "median", "std", "min", "max", "skew"
]).to_frame("target")

display(target_stats)

q1 = df["target"].quantile(0.25)
q3 = df["target"].quantile(0.75)
iqr = q3 - q1
limite_inf = q1 - 1.5 * iqr
limite_sup = q3 + 1.5 * iqr

outliers = df[(df["target"] < limite_inf) | (df["target"] > limite_sup)]

print(f"Q1: {q1:.6f}")
print(f"Q3: {q3:.6f}")
print(f"IQR: {iqr:.6f}")
print(f"Limite superior pelo critério 1,5*IQR: {limite_sup:.6f}")
print(f"Possíveis outliers: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["target"].dropna(), bins=80)
plt.title("Distribuição da volatilidade realizada (target)")
plt.xlabel("Volatilidade realizada")
plt.ylabel("Frequência")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(df["target"].dropna(), vert=False)
plt.title("Boxplot da volatilidade realizada")
plt.xlabel("Volatilidade realizada")
plt.grid(alpha=0.2)
plt.show()

### Interpretação esperada

Compare principalmente **média e mediana**. Se a média for consideravelmente maior que a mediana e a assimetria for positiva, isso indica que a distribuição possui uma cauda à direita, causada por períodos de volatilidade excepcionalmente elevada.

O boxplot permite verificar observações afastadas da região central. No contexto financeiro, esses pontos não devem ser automaticamente tratados como erros: eles podem representar episódios legítimos de maior instabilidade.

## 8. Volatilidade por ação

Agora investigamos se diferentes ações apresentam comportamentos distintos.

In [ ]:
por_acao = (
    df.groupby("stock_id")["target"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .sort_values("mean", ascending=False)
)

por_acao.head(15)

In [ ]:
top15 = por_acao.head(15).sort_values("mean")

plt.figure(figsize=(10, 6))
plt.barh(top15.index.astype(str), top15["mean"])
plt.title("15 ações com maior volatilidade média")
plt.xlabel("Volatilidade média")
plt.ylabel("stock_id")
plt.grid(axis="x", alpha=0.2)
plt.show()

In [ ]:
bottom15 = por_acao.sort_values("mean").head(15)

plt.figure(figsize=(10, 6))
plt.barh(bottom15.index.astype(str), bottom15["mean"])
plt.title("15 ações com menor volatilidade média")
plt.xlabel("Volatilidade média")
plt.ylabel("stock_id")
plt.grid(axis="x", alpha=0.2)
plt.show()

### Interpretação

A comparação entre ações permite verificar se a volatilidade é homogênea ou se existem ativos persistentemente mais instáveis. Diferenças relevantes entre médias, medianas e desvios-padrão sugerem que o identificador da ação contém informação importante para modelos futuros.

## 9. Comportamento por intervalo temporal

Embora `time_id` não represente diretamente uma data/hora tradicional, ele identifica janelas temporais consistentes entre diferentes ações. Podemos utilizá-lo para verificar se determinados intervalos apresentam maior volatilidade de mercado.

In [ ]:
por_tempo = (
    df.groupby("time_id")["target"]
      .agg(["count", "mean", "median", "std"])
      .sort_index()
)

display(por_tempo.head())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(por_tempo.index, por_tempo["mean"], linewidth=0.8)
plt.title("Volatilidade média por time_id")
plt.xlabel("time_id")
plt.ylabel("Volatilidade média")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
top_tempos = por_tempo.sort_values("mean", ascending=False).head(15)
display(top_tempos)

### Interpretação

Picos na volatilidade média em determinados `time_id` indicam períodos nos quais diversas ações sofreram maior instabilidade simultaneamente. Isso pode sugerir a existência de fatores sistêmicos ou eventos de mercado afetando vários ativos.

## 10. Distribuição das observações por ação

In [ ]:
obs_por_acao = df["stock_id"].value_counts().sort_index()

print(obs_por_acao.describe())

plt.figure(figsize=(12, 5))
plt.bar(obs_por_acao.index.astype(str), obs_por_acao.values)
plt.title("Quantidade de observações por ação")
plt.xlabel("stock_id")
plt.ylabel("Quantidade de registros")
plt.xticks([])
plt.grid(axis="y", alpha=0.2)
plt.show()

Esta análise permite verificar se todas as ações possuem a mesma quantidade de períodos observados. Diferenças no número de registros podem influenciar comparações e devem ser consideradas nas etapas de modelagem.

## 11. Análise dos dados de livro de ofertas — amostra de uma ação

Para enriquecer a EDA sem carregar todo o conjunto em memória, vamos abrir apenas uma partição de `book_train.parquet`.

O livro de ofertas contém preços e volumes das melhores ofertas de compra (`bid`) e venda (`ask`) do mercado.

In [ ]:
# Procura uma partição do book_train
book_files = glob.glob("/content/optiver/**/book_train.parquet/**/*.parquet", recursive=True)

print("Partições encontradas:", len(book_files))

if book_files:
    book_file = book_files[0]
    print("Amostra utilizada:", book_file)
    book = pd.read_parquet(book_file)
    display(book.head())
    print(book.shape)
else:
    book = None
    print("Nenhuma partição book_train encontrada. As análises principais com train.csv continuam válidas.")

In [ ]:
if book is not None:
    display(book.describe().T)
    print("\nAusentes:")
    display(book.isna().sum().to_frame("ausentes"))

## 12. Engenharia de variáveis exploratórias do livro de ofertas

Duas medidas úteis são:

**WAP (Weighted Average Price)** — preço médio ponderado utilizando preço e tamanho das ofertas.

**Spread** — diferença entre o melhor preço de venda e o melhor preço de compra.

Também calcularemos o retorno logarítmico do WAP.

In [ ]:
def log_return(series):
    return np.log(series).diff()

if book is not None:
    book = book.copy()

    book["wap"] = (
        book["bid_price1"] * book["ask_size1"] +
        book["ask_price1"] * book["bid_size1"]
    ) / (book["bid_size1"] + book["ask_size1"])

    book["spread"] = book["ask_price1"] - book["bid_price1"]

    book = book.sort_values(["time_id", "seconds_in_bucket"])
    book["log_return"] = book.groupby("time_id")["wap"].transform(log_return)

    display(book[[
        "time_id", "seconds_in_bucket", "bid_price1", "ask_price1",
        "bid_size1", "ask_size1", "wap", "spread", "log_return"
    ]].head(15))

## 13. Volatilidade realizada calculada a partir dos retornos

A volatilidade realizada em uma janela pode ser calculada como:

\[
RV = \sqrt{\sum_{t=1}^{n} r_t^2}
\]

em que `r_t` representa o retorno logarítmico observado no instante `t`.

In [ ]:
def realized_volatility(series):
    return np.sqrt(np.sum(series.dropna() ** 2))

if book is not None:
    rv_book = (
        book.groupby("time_id")["log_return"]
            .apply(realized_volatility)
            .reset_index(name="realized_volatility_book")
    )

    display(rv_book.head())
    display(rv_book["realized_volatility_book"].describe())

In [ ]:
if book is not None:
    plt.figure(figsize=(10, 5))
    plt.hist(rv_book["realized_volatility_book"], bins=60)
    plt.title("Distribuição da volatilidade calculada na amostra do order book")
    plt.xlabel("Volatilidade realizada")
    plt.ylabel("Frequência")
    plt.grid(alpha=0.2)
    plt.show()

## 14. Relação entre spread e volatilidade

Nesta seção agregamos o spread médio por janela temporal e verificamos sua relação com a volatilidade calculada.

In [ ]:
if book is not None:
    features_book = (
        book.groupby("time_id")
            .agg(
                spread_medio=("spread", "mean"),
                wap_medio=("wap", "mean"),
                volume_bid_medio=("bid_size1", "mean"),
                volume_ask_medio=("ask_size1", "mean")
            )
            .reset_index()
            .merge(rv_book, on="time_id", how="left")
    )

    display(features_book.head())
    display(features_book.corr(numeric_only=True))

In [ ]:
if book is not None:
    plt.figure(figsize=(8, 5))
    plt.scatter(
        features_book["spread_medio"],
        features_book["realized_volatility_book"],
        alpha=0.4,
        s=12
    )
    plt.title("Spread médio x volatilidade realizada")
    plt.xlabel("Spread médio")
    plt.ylabel("Volatilidade realizada")
    plt.grid(alpha=0.2)
    plt.show()

### Interpretação

Uma associação positiva entre spread e volatilidade seria compatível com períodos de maior incerteza, nos quais a distância entre preços de compra e venda pode aumentar. Entretanto, correlação não implica causalidade; o resultado deve ser interpretado apenas como uma associação exploratória.

## 15. Dados de trades — amostra de uma ação

In [ ]:
trade_files = glob.glob("/content/optiver/**/trade_train.parquet/**/*.parquet", recursive=True)

print("Partições encontradas:", len(trade_files))

if trade_files:
    # tenta encontrar a mesma stock_id da partição do book
    if book_files:
        stock_folder = os.path.basename(os.path.dirname(book_file))
        same_stock = [p for p in trade_files if stock_folder in p]
        trade_file = same_stock[0] if same_stock else trade_files[0]
    else:
        trade_file = trade_files[0]

    print("Amostra utilizada:", trade_file)
    trade = pd.read_parquet(trade_file)
    display(trade.head())
    print(trade.shape)
else:
    trade = None
    print("Nenhuma partição trade_train encontrada.")

In [ ]:
if trade is not None:
    display(trade.describe().T)

    trades_agg = (
        trade.groupby("time_id")
             .agg(
                 preco_medio=("price", "mean"),
                 volume_total=("size", "sum"),
                 ordens_total=("order_count", "sum"),
                 qtd_negocios=("price", "count")
             )
             .reset_index()
    )

    display(trades_agg.head())

## 16. Relações multivariadas na amostra

Vamos combinar atributos do livro de ofertas, trades e volatilidade realizada para verificar associações.

In [ ]:
if book is not None and trade is not None:
    multi = features_book.merge(trades_agg, on="time_id", how="inner")

    corr = multi.corr(numeric_only=True)

    display(corr)

    plt.figure(figsize=(9, 7))
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlação")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Matriz de correlação — variáveis exploratórias")
    plt.tight_layout()
    plt.show()

## 17. Identificação de padrões e anomalias

Nesta etapa reunimos alguns indicadores objetivos:

- ações com maior volatilidade média;
- períodos com maior volatilidade média;
- proporção de possíveis outliers segundo o critério do IQR;
- dispersão da volatilidade entre ações.

In [ ]:
print("=== RESUMO EXPLORATÓRIO ===")
print(f"Quantidade de ações: {df['stock_id'].nunique()}")
print(f"Quantidade de time_id: {df['time_id'].nunique()}")
print(f"Volatilidade média: {df['target'].mean():.6f}")
print(f"Volatilidade mediana: {df['target'].median():.6f}")
print(f"Desvio-padrão: {df['target'].std():.6f}")
print(f"Assimetria: {df['target'].skew():.4f}")
print(f"Possíveis outliers por IQR: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

print("\n5 ações com maior volatilidade média:")
display(por_acao.head(5)[["mean", "median", "std"]])

print("\n5 períodos com maior volatilidade média:")
display(por_tempo.sort_values("mean", ascending=False).head(5)[["mean", "median", "std"]])

## 18. Insights

Após executar todas as células, utilize os resultados acima para confirmar os pontos abaixo.

### Insight 1 — A volatilidade não é necessariamente distribuída de forma simétrica

Caso a média seja superior à mediana e a assimetria seja positiva, a distribuição possui concentração em valores baixos/moderados e uma cauda formada por períodos de maior volatilidade.

### Insight 2 — Existem diferenças entre as ações

O ranking de volatilidade média permite observar se certos `stock_id` apresentam comportamento sistematicamente mais volátil que outros. Isso sugere que características específicas dos ativos são relevantes para a previsão.

### Insight 3 — Existem períodos de estresse compartilhados

Picos da volatilidade média agrupada por `time_id` indicam janelas em que vários ativos apresentam aumento simultâneo de instabilidade.

### Insight 4 — Valores extremos merecem investigação, não remoção automática

Em séries financeiras, observações extremas podem representar eventos reais de mercado. Assim, a decisão de removê-las apenas com base no IQR seria inadequada sem uma investigação adicional.

### Insight 5 — Microestrutura pode conter informação sobre volatilidade

Na amostra do livro de ofertas, variáveis como spread, volumes de compra/venda e atividade de trades podem apresentar associação com a volatilidade. Essas relações são candidatas naturais para criação de atributos em uma futura etapa de modelagem.

## 19. Conclusão

A Análise Exploratória permitiu compreender a estrutura inicial da base Optiver e investigar a volatilidade realizada de diferentes ações.

Foram avaliados:

- dimensão e estrutura do conjunto;
- tipos e qualidade dos dados;
- valores ausentes e duplicações;
- estatísticas descritivas da variável-alvo;
- distribuição e possíveis valores atípicos;
- diferenças de volatilidade entre ações;
- comportamento da volatilidade entre diferentes intervalos;
- informações do livro de ofertas e trades em uma amostra;
- possíveis associações entre variáveis de microestrutura e volatilidade.

A análise evidencia a importância de considerar tanto **características específicas de cada ativo** quanto **mudanças ao longo dos intervalos de mercado**. A presença de valores extremos e possíveis distribuições assimétricas também indica que métricas robustas e transformações podem ser importantes em etapas posteriores.

### Limitações

1. `stock_id` não revela a identidade real das empresas.
2. `time_id` identifica intervalos, mas não fornece diretamente data e horário convencional.
3. A análise detalhada dos arquivos Parquet foi realizada por amostragem para evitar consumo excessivo de memória.
4. Relações observadas nesta EDA representam associações e não causalidade.

### Próximos passos

Como continuidade do projeto, recomenda-se:

1. construir atributos agregados do order book e de trades para todas as ações;
2. calcular medidas adicionais de liquidez, spread e volume;
3. criar uma estratégia de validação temporal;
4. testar modelos de regressão e forecasting;
5. avaliar as previsões usando a métrica definida no projeto da disciplina.

## 20. Checklist da rubrica

- [x] Contextualização do problema
- [x] Objetivo da análise
- [x] Origem dos dados
- [x] Descrição das observações e variáveis
- [x] Número de linhas e colunas
- [x] Tipos das variáveis
- [x] Estatísticas descritivas
- [x] Valores ausentes
- [x] Duplicações
- [x] Valores inválidos
- [x] Possíveis outliers
- [x] Análise univariada
- [x] Análise bivariada/multivariada
- [x] Visualizações
- [x] Padrões, tendências e anomalias
- [x] Insights
- [x] Conclusão
- [x] Limitações
- [x] Próximos passos